In [1]:
!pip install xarray
!pip install pandas
!pip install ecmwf-opendata
!pip install hvplot
!pip install cfgrib

In [2]:
import xarray as xr
import pandas as pd
from ecmwf.opendata import Client
import hvplot.xarray

%opts magic unavailable (pyparsing cannot be imported)
%compositor magic unavailable (pyparsing cannot be imported)


In [3]:

client = Client("ecmwf", beta=False)
parameters = ['tp']
filename = 'medium-rain-acc.grib'
filename

'medium-rain-acc.grib'

In [4]:

client.retrieve(
    step=240,
    stream="oper",
    type="fc",
    levtype="sfc",
    resol='0p25',
    param=parameters,
    target=filename
)



20250402000000-240h-oper-fc.grib2:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

In [5]:
euro = xr.open_dataset(filename, engine='cfgrib')

/srv/conda/envs/notebook/lib/python3.10/site-packages/cfgrib/xarray_plugin.py:131: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


In [6]:
df = pd.read_csv(r'acres.csv')

In [7]:
!pip install geopandas
!pip install geoviews

import geopandas as gpd
import geoviews as gv

counties = gpd.read_file(r'https://gist.githubusercontent.com/sdwfrost/d1c73f91dd9d175998ed166eb216994a/raw/e89c35f308cee7e2e5a784e1d3afc5d449e9e4bb/counties.geojson')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.9/23.9 MB 51.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.3/547.3 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.6 MB/s eta 0:00:00


In [9]:
counties.GEOID = counties.GEOID.astype(str)
df.ANSI = df.ANSI.astype(str)

joint = counties.set_index('GEOID').join(df.set_index('ANSI')).dropna(subset=['VALUE'])

In [10]:
joint.sort_values('VALUE', ascending=False, inplace=True)
joint.rename(columns={'VALUE':'RAINFED_ACRES'}, inplace=True)

In [13]:
!pip install scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.6/37.6 MB 63.3 MB/s eta 0:00:00a 0:00:01


In [14]:
(euro['tp']*39.37).hvplot(height=600, width=1250, coastline=True, features={'states':'50m'}) * gv.Polygons(joint[['LOCATION_DESC','RAINFED_ACRES','geometry']]).opts(color='RAINFED_ACRES',alpha=0.25, cmap='kgy')

/srv/conda/envs/notebook/lib/python3.10/site-packages/cartopy/io/__init__.py:241: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_1_states_provinces_lakes.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


:Overlay
   .Image.I     :Image   [longitude,latitude]   (tp)
   .Coastline.I :Feature   [Longitude,Latitude]
   .States.I    :Feature   [Longitude,Latitude]
   .Polygons.I  :Polygons   [Longitude,Latitude]   (LOCATION_DESC,RAINFED_ACRES)